In [0]:
%run ./utils

In [0]:
from datetime import datetime, timedelta
import pyspark.sql.functions as F

def get_checksum_df(clear_consumer_table, landing_consumer_path, start_time, end_time):

    landing_df = (spark.read.format("delta").load(landing_consumer_path)
        .filter((F.col("slndc_update_dt") >= F.lit(start_time)) & (F.col("slndc_update_dt") <= F.lit(end_time)))
        .select(
            F.col("task_id"),
            F.col("slndc_batch_id").alias("batch_id"),
            F.col("slndc_batch_dt").alias("batch_dt"),
            F.col("slndc_kafka_topic").alias("kafka_topic"),
            F.col("slndc_update_dt"),
            F.col("batch_number")
        )
        .distinct()
    )

    clear_df = (spark.table(clear_consumer_table)
        .select("batch_number")
        .distinct())

    check_df = (landing_df.alias("landing_df")
        .join(clear_df.alias("clear_df"), 
            (F.col("landing_df.batch_number") == F.col("clear_df.batch_number")),
            "left"
        )
        .filter(F.col("clear_df.batch_number").isNull())
        .select(F.col("landing_df.*"))
        .distinct()
    )

    return check_df


In [0]:
def monitor_main(monitor_id, clear_consumer_table, landing_consumer_path, start_time, end_time, max_rows=MAX_ROWS, max_cols=MAX_COLS, to_addrs=None):
    # 1. check sum
    check_df = get_checksum_df(clear_consumer_table, landing_consumer_path, start_time, end_time)
    check_df.cache()

    if check_df.count() > 0:
        print(f"This inspection found invalid data: {monitor_id}")
        display(check_df)

        # 2. build email body
        html_body = build_html_table_from_spark_df(check_df, max_rows=max_rows, max_cols=max_cols)

        recipients = to_addrs or TO_ADDRS
        if not recipients:
            raise ValueError("to_addrs is empty; no recipients configured for the batch number missing report email.")

        send_email(
            subject=SUBJECT.format(yyyymmdd=end_time.strftime("%Y%m%d")),
            html_body=html_body,
            to_addrs=recipients,
            cc_addrs=CC_ADDRS,
            bcc_addrs=BCC_ADDRS,
            custom_text = f"This inspection found miss BatchNum.  <br>Check time period(UTC): {start_time} -> {end_time}. <br>Check the table: {landing_consumer_path} -> {clear_consumer_table}. <br>monitor_id: {monitor_id}"
        )

    else:
        print(f"There is no invalid data in this check: {monitor_id}")

    check_df.unpersist()

In [0]:
TO_ADDRS: List[str] = []
CC_ADDRS: List[str] = []
BCC_ADDRS: List[str] = []

SUBJECT = "[Major] [MDM] Batch Not Process {yyyymmdd}"

In [0]:
monitor_id = dbutils.widgets.get("monitor_id")
hour_time_period = int(dbutils.widgets.get("hour_time_period"))
clear_consumer_table = dbutils.widgets.get("clear_consumer_table")
landing_consumer_path = dbutils.widgets.get("landing_consumer_path")

try:
    trigger_timestamp_ms = int(dbutils.widgets.get("trigger_timestamp_ms")) / 1000
except:
    trigger_timestamp_ms = int(datetime.now().timestamp())

# Maximum rows/columns to show in the email HTML tables.
try:
    max_rows = int(dbutils.widgets.get("max_rows"))
except:
    max_rows = MAX_ROWS

try:
    max_cols = int(dbutils.widgets.get("max_cols"))
except:
    max_cols = MAX_COLS

# Comma-separated list of recipient email addresses.
to_addrs_str = dbutils.widgets.get("to_addrs")
to_addrs = [x.strip() for x in to_addrs_str.split(",") if x.strip()] if to_addrs_str else TO_ADDRS

end_time = datetime.fromtimestamp(trigger_timestamp_ms)
start_time = end_time - timedelta(hours= hour_time_period)

print(f"monitor_id: {monitor_id}")
print(f"clear_consumer_table: {clear_consumer_table}")
print(f"landing_consumer_path: {landing_consumer_path}")
print(f"hour_time_period: {hour_time_period}")
print(f"max_rows: {max_rows}, max_cols: {max_cols}")
print(f"to_addrs: {to_addrs}")
print(f"start_time: {start_time}, end_time: {end_time}")

monitor_main(monitor_id, clear_consumer_table, landing_consumer_path, start_time, end_time, max_rows, max_cols, to_addrs)